# 新版 LangChain：通过 Token 限制 Memory

这一节对应老版本的 `ConversationTokenBufferMemory`。

> 目标：让历史消息总 token 不超过阈值，避免上下文无限增长。

## 新旧版本映射

- 旧版：`ConversationTokenBufferMemory(llm=..., max_token_limit=...)`
- 新版：`RunnableWithMessageHistory` + `trim_messages(...)`

通常有两种策略：

1. **仅限制注入给模型的历史**（存储可保留全量）
2. **连存储一起裁剪到 token 上限**（更接近旧版行为）

In [1]:
# 1) 环境准备
import os
from typing import Dict, List

from dotenv import load_dotenv
from pydantic import SecretStr
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory

# 不同版本 langchain-core 可能导入路径略有差异
try:
    from langchain_core.messages import trim_messages
except ImportError:
    from langchain_core.messages.utils import trim_messages  # type: ignore

load_dotenv(override=True)

model_name = os.getenv("MODEL")
model_base_url = os.getenv("BASE_URL")
model_api_key = os.getenv("API_KEY")
api_key_value = SecretStr(model_api_key) if model_api_key else None

chat_llm = ChatOpenAI(
    model=model_name or "gpt-4o-mini",
    base_url=model_base_url or None,
    api_key=api_key_value,
    temperature=0,
 )

print("✅ 模型初始化完成")

✅ 模型初始化完成


In [6]:
# 2) 方案A：只限制“注入给模型”的历史 token（推荐起步）
MAX_HISTORY_TOKENS = 200
SYSTEM_PROMPT = "你是一个简洁的中文助理。"

prompt_token_window = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

def count_tokens(messages: List) -> int:
    """统计消息 token（基于当前模型适配器的计数实现）。"""
    return chat_llm.get_num_tokens_from_messages(messages)

def build_prompt_messages(history: List, user_input: str):
    """构造实际发给模型的消息列表（system + history + 当前input）。"""
    return prompt_token_window.format_messages(history=history, input=user_input)

def trim_for_prompt(payload: Dict):
    full_history = payload.get("history", [])
    trimmed_history = trim_messages(
        messages=full_history,
        token_counter=chat_llm,
        max_tokens=MAX_HISTORY_TOKENS,
        strategy="last",
        include_system=True,
        start_on="human",
        allow_partial=False,
    )
    return {"input": payload["input"], "history": trimmed_history}

base_chain_token_window = RunnableLambda(trim_for_prompt) | prompt_token_window | chat_llm

store_token_window: Dict[str, ChatMessageHistory] = {}

def get_token_window_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store_token_window:
        store_token_window[session_id] = ChatMessageHistory()
    return store_token_window[session_id]

chain_with_token_window = RunnableWithMessageHistory(
    base_chain_token_window,
    get_token_window_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("✅ 方案A链创建完成")

✅ 方案A链创建完成


In [7]:
# 3) 运行多轮，观察：存储可增长，但每次注入历史受 token 上限约束
session_a = "token_window_user_001"
inputs_a = [
    "你好，我叫小严。",
    "我在学 LangChain 的 memory。",
    "我想知道 RunnableWithMessageHistory 怎么工作。",
    "再解释一下为什么要做 token 限制。",
    "你还记得我叫什么吗？",
]

for i, q in enumerate(inputs_a, start=1):
    out = chain_with_token_window.invoke(
        {"input": q},
        config={"configurable": {"session_id": session_a}},
    )
    print(f"第{i}轮回答: {out.content}")

all_history_a = store_token_window[session_a].messages
trimmed_a = trim_messages(
    messages=all_history_a,
    token_counter=chat_llm,
    max_tokens=MAX_HISTORY_TOKENS,
    strategy="last",
    include_system=True,
    start_on="human",
    allow_partial=False,
)

# 用最后一轮问题，构造真实发送给模型的消息，计算“总prompt token”
last_input_a = inputs_a[-1]
final_prompt_a = build_prompt_messages(trimmed_a, last_input_a)

print("\n存储总消息条数:", len(all_history_a))
print("存储总token数(历史全量):", count_tokens(all_history_a))
print("注入窗口消息条数:", len(trimmed_a))
print("注入窗口token数(仅history):", count_tokens(trimmed_a), f"/ 目标上限 {MAX_HISTORY_TOKENS}")
print("最后一轮总prompt token(system+history+input):", count_tokens(final_prompt_a))
print("\n说明：MAX_HISTORY_TOKENS 只限制 history 注入窗口，不限制全量存储。")

第1轮回答: 你好，小严，很高兴认识你！有什么我可以帮你的？
第2轮回答: 很棒，小严！LangChain 的 **memory** 主要是让模型“记住”上下文，常见于聊天机器人和多轮对话。

你可以先把它理解成三类：

1. **短期记忆**：保留最近几轮对话  
2. **长期记忆**：保存用户偏好、关键信息  
3. **检索式记忆**：把历史内容存到向量库，需要时再取出来

如果你愿意，我可以接着帮你：
- 用 **最简单的代码** 讲清楚 memory 怎么用
- 对比 **ConversationBufferMemory / SummaryMemory / VectorStoreMemory**
- 按你正在用的 **LangChain 版本** 讲最新写法

你现在最想学哪一种？
第3轮回答: `RunnableWithMessageHistory` 可以理解成：**给任意 Runnable 自动加上“按会话保存/读取消息历史”的外壳**。

## 它怎么工作

一次调用时，大致流程是这样的：

1. **从配置里拿到会话标识**  
   例如 `session_id="abc"`。

2. **根据 session_id 取出历史消息对象**  
   你提供一个函数，比如：
   ```python
   get_session_history(session_id) -> BaseChatMessageHistory
   ```

3. **把历史消息注入到输入里**
   - 如果你的 chain 需要 `history` 这个字段，它就把历史放进去
   - 如果你的 runnable 直接吃消息列表，它就把“历史 + 本轮消息”拼起来

4. **执行原始 Runnable**

5. **把本轮用户输入 + 模型输出写回历史**
   这样下次同一个 session 再调用时，就能继续沿用上下文。

---

## 核心理解

你可以把它想成：

> **“每次调用前先读历史，调用后再写历史。”**

它本身**不负责存储**，只是负责“接线”。  
真正的存储由你提供的 `BaseChatMessageHistory` 实现负责，比如：
- `InMemoryChatMessageHistory`
- Redis
- 数据库
- 文

## 方案B：连“存储”也按 token 裁剪（更接近旧版 ConversationTokenBufferMemory）

如果你想完全模拟旧版行为，可在每次调用后把历史裁剪回 token 上限。

In [4]:
# 4) 方案B实现：每轮调用后，强制把存储裁剪回 token 上限
store_token_strict: Dict[str, ChatMessageHistory] = {}

def get_token_strict_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store_token_strict:
        store_token_strict[session_id] = ChatMessageHistory()
    return store_token_strict[session_id]

prompt_token_strict = ChatPromptTemplate.from_messages([
    ("system", "你是一个简洁的中文助理。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

base_chain_token_strict = prompt_token_strict | chat_llm

chain_with_token_strict = RunnableWithMessageHistory(
    base_chain_token_strict,
    get_token_strict_history,
    input_messages_key="input",
    history_messages_key="history",
)

def prune_store_by_tokens(session_id: str):
    history = store_token_strict[session_id]
    pruned = trim_messages(
        messages=history.messages,
        token_counter=chat_llm,
        max_tokens=MAX_HISTORY_TOKENS,
        strategy="last",
        include_system=True,
        start_on="human",
        allow_partial=False,
    )
    history.messages = list(pruned)

def invoke_with_token_strict(user_input: str, session_id: str):
    result = chain_with_token_strict.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}},
    )
    prune_store_by_tokens(session_id)
    return result

print("✅ 方案B链创建完成")

✅ 方案B链创建完成


In [5]:
# 5) 验证方案B：存储token会被限制在阈值内
session_b = "token_strict_user_001"
inputs_b = [
    "第一轮：我叫小严。",
    "第二轮：我来自北京。",
    "第三轮：我喜欢篮球，也喜欢羽毛球。",
    "第四轮：我最近在学习 LangChain 的 memory 机制。",
    "第五轮：你还记得我是谁、来自哪里吗？",
]

for i, q in enumerate(inputs_b, start=1):
    out = invoke_with_token_strict(q, session_b)
    tokens_now = count_tokens(store_token_strict[session_b].messages)
    print(f"第{i}轮 -> 存储token: {tokens_now} / {MAX_HISTORY_TOKENS}")
    print("回答:", out.content)
    print("-" * 30)

print("最终存储消息条数:", len(store_token_strict[session_b].messages))
print("最终存储token:", count_tokens(store_token_strict[session_b].messages))

第1轮 -> 存储token: 28 / 200
回答: 你好，小严，很高兴认识你。
------------------------------
第2轮 -> 存储token: 59 / 200
回答: 北京是个很棒的地方，小严。很高兴了解你。
------------------------------
第3轮 -> 存储token: 100 / 200
回答: 挺好的，小严！篮球和羽毛球都很有意思，也很锻炼身体。
------------------------------
第4轮 -> 存储token: 173 / 200
回答: 不错，小严！LangChain 的 memory 机制很重要，主要是用来让模型记住上下文、保存对话状态。  
如果你愿意，我可以继续帮你梳理它的几种常见实现方式。
------------------------------
第5轮 -> 存储token: 181 / 200
回答: 当然记得，你叫小严，来自北京。
------------------------------
最终存储消息条数: 8
最终存储token: 181


## 小结

- 你之前的 `ConversationTokenBufferMemory`，在新版中可以理解为：
  - `RunnableWithMessageHistory` 负责记忆接入
  - `trim_messages(..., max_tokens=...)` 负责 token 控制
- 想“最小改造”：用方案A。
- 想“完全贴近旧版行为”：用方案B（调用后裁剪存储）。